# V2.5.2 — Fair Comparison: XGBoost vs LightGBM (both Optuna-tuned, MAE loss)

Using exactly the same data, the MAE loss function, and the Optuna search process (10 runs × 5 folds),
we conducted a fair comparison to determine whether XGBoost or LightGBM is better suited for predicting 15-minute electricity prices in Finland.

- Similarities: Data, 80/20 split, MAE loss, Optuna search process, number of trees
- Differences: Only the algorithms themselves (XGBoost’s level-wise approach vs. LightGBM’s leaf-wise approach)

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

optuna.logging.set_verbosity(optuna.logging.WARNING)

e:\Github\nordpool_electricity_price_prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load the dataset
df = pd.read_csv('../data/convertData/V2.5_15min_features.csv')
print(df.shape)
df.head()

(105216, 51)


,datetime,price,temp,wind_speed,wind_direction_deg,wind_dir_sin,wind_dir_cos,hour,minute,day_of_week,...,price_rolling_mean_24h,price_rolling_std_24h,price_rolling_min_24h,price_rolling_max_24h,price_rolling_mean_7d,temp_rolling_mean_1h,HDD,wind_power_proxy,temp_lag_4,temp_lag_96
0,2023-01-01 00:00:00+02:00,4.8400,4.45,8.45,215.35,-0.578569,-0.815633,0,0,6,...,NaN,NaN,NaN,NaN,NaN,NaN,12.55,603.351125,NaN,NaN
1,2023-01-01 00:15:00+02:00,4.1325,4.50,9.00,213.10,-0.546102,-0.837719,0,15,6,...,NaN,NaN,NaN,NaN,NaN,NaN,12.50,729.000000,NaN,NaN
2,2023-01-01 00:30:00+02:00,3.4250,4.45,8.55,211.25,-0.518650,-0.854708,0,30,6,...,NaN,NaN,NaN,NaN,NaN,NaN,12.55,625.026375,NaN,NaN
3,2023-01-01 00:45:00+02:00,2.7175,4.30,8.10,209.70,-0.495459,-0.868632,0,45,6,...,NaN,NaN,NaN,NaN,NaN,NaN,12.70,531.441000,NaN,NaN
4,2023-01-01 01:00:00+02:00,2.0100,4.30,8.10,211.85,-0.527451,-0.849036,1,0,6,...,NaN,NaN,NaN,NaN,NaN,4.425,12.70,531.441000,4.45,NaN


In [ ]:
#  80/20 split in chronological order (no shuffling - time series data, future cannot leak into training)
X = df.drop(columns=['price', 'datetime'])
y = df['price']

n = len(df)
test_size = int(n * 0.20)
train_end = n - test_size

X_train = X.iloc[:train_end];  y_train = y.iloc[:train_end]
X_test  = X.iloc[train_end:];  y_test  = y.iloc[train_end:]
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

Train: (84173, 49)  Test: (21043, 49)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# core: one code to run two models fairly
# - same search space (common part)
# - same TimeSeriesSplit(5) folds
# - same evaluation (5-fold mean MAE)
# - only difference: each model uses its own native parameters + algorithm itself   
# ═══════════════════════════════════════════════════════════════════════
N_TRIALS = 10          # fast validation (same for both models)
N_ESTIMATORS = 2000    # same number of trees (same for both models)

tscv = TimeSeriesSplit(n_splits=5)   # same 5-fold split object

def make_objective(model_type):
    """返回一个针对指定模型的 Optuna objective 函数。"""
    def objective(trial):
        # two models share the same search space (exactly the same)
        common = {
            'n_estimators': N_ESTIMATORS,
            'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_lambda':       trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),
            'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 1.0),
            'random_state': 42,
        }
        if model_type == 'xgb':
            params = {**common,
                      'objective': 'reg:absoluteerror',          # MAE 损失
                      'max_depth':        trial.suggest_int('max_depth', 4, 12),
                      'min_child_weight': trial.suggest_int('min_child_weight', 1, 50)}
            model = XGBRegressor(**params, verbosity=0)
        else:
            params = {**common,
                      'objective': 'regression_l1',              # MAE 损失
                      'num_leaves':        trial.suggest_int('num_leaves', 63, 511),
                      'min_child_samples': trial.suggest_int('min_child_samples', 10, 100)}
            model = lgb.LGBMRegressor(**params, verbose=-1)

        # 5-fold cross-validation to evaluate the model
        fold_maes = []
        for train_idx, val_idx in tscv.split(X_train):
            Xf, yf = X_train.iloc[train_idx], y_train.iloc[train_idx]
            Xv, yv = X_train.iloc[val_idx], y_train.iloc[val_idx]
            model.fit(Xf, yf)
            fold_maes.append(mean_absolute_error(yv, model.predict(Xv)))
        return np.mean(fold_maes)
    return objective

In [5]:
# ── XGBoost optuna search ─────────────────────────────────────────────
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(make_objective('xgb'), n_trials=N_TRIALS, show_progress_bar=True)
print(f'XGBoost best CV MAE : {study_xgb.best_value:.4f}')
print(f'XGBoost best params : {study_xgb.best_params}')

Best trial: 8. Best value: 2.99524: 100%|██████████| 10/10 [17:49<00:00, 106.91s/it]

XGBoost best CV MAE : 2.9952
XGBoost best params : {'learning_rate': 0.05901799493774904, 'subsample': 0.7268039180070345, 'colsample_bytree': 0.9753321409349, 'reg_lambda': 0.3615584631348949, 'reg_alpha': 0.6781619640418217, 'max_depth': 7, 'min_child_weight': 21}


In [6]:
# ── LightGBM optuna search ─────────────────────────────────────────────
study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(make_objective('lgb'), n_trials=N_TRIALS, show_progress_bar=True)
print(f'LightGBM best CV MAE: {study_lgb.best_value:.4f}')
print(f'LightGBM best params: {study_lgb.best_params}')

Best trial: 3. Best value: 2.88508: 100%|██████████| 10/10 [16:19<00:00, 97.91s/it]

LightGBM best CV MAE: 2.8851
LightGBM best params: {'learning_rate': 0.016783049642310737, 'subsample': 0.9914750371619152, 'colsample_bytree': 0.9784254502903834, 'reg_lambda': 0.28259635343517275, 'reg_alpha': 0.23501192153836958, 'num_leaves': 313, 'min_child_samples': 36}


In [7]:
# Train final models with optimal parameters
model_xgb = XGBRegressor(objective='reg:absoluteerror',
                         n_estimators=N_ESTIMATORS, random_state=42,
                         **study_xgb.best_params, verbosity=0)
model_xgb.fit(X_train, y_train)

model_lgb = lgb.LGBMRegressor(objective='regression_l1',
                              n_estimators=N_ESTIMATORS, random_state=42,
                              **study_lgb.best_params, verbose=-1)
model_lgb.fit(X_train, y_train)
print('Both final models trained.')

Both final models trained.


In [ ]:
# Evaluate models 
def evaluate(model):
    pred = model.predict(X_test)
    return (mean_absolute_error(y_test, pred),
            np.sqrt(mean_squared_error(y_test, pred)),
            r2_score(y_test, pred))

xgb_mae, xgb_rmse, xgb_r2 = evaluate(model_xgb)
lgb_mae, lgb_rmse, lgb_r2 = evaluate(model_lgb)

comparison = pd.DataFrame({
    'Model':  ['XGBoost V2.5.2', 'LightGBM V2.5.2'],
    'MAE':    [xgb_mae, lgb_mae],
    'RMSE':   [xgb_rmse, lgb_rmse],
    'R²':     [xgb_r2,  lgb_r2],
}).set_index('Model').round(4)
print(comparison)

                MAE    RMSE      R²
Model                              
XGBoost V2.5.2   2.7652  8.2342  0.9717
LightGBM V2.5.2  2.7167  8.0958  0.9727


In [ ]:
# save the models and feature columns for future use
import joblib
from pathlib import Path

save_dir = Path('../models/saved')
save_dir.mkdir(exist_ok=True)

joblib.dump({'model': model_xgb, 'feature_cols': X_train.columns.tolist(), 'step_min': 15},
            save_dir / 'xgboost_v2_5_2.pkl')
joblib.dump({'model': model_lgb, 'feature_cols': X_train.columns.tolist(), 'step_min': 15},
            save_dir / 'lightgbm_v2_5_2.pkl')
print('Saved xgboost_v2_5_2.pkl and lightgbm_v2_5_2.pkl')

Saved xgboost_v2_5_2.pkl and lightgbm_v2_5_2.pkl
